# NDR — Notebook 3: PRUNE-ONLY De-Poisoning (không fine-tune)

**Deliverable:** model đã sửa weight bằng *pruning phẫu thuật* các neuron nhạy poison. Submit output TRỰC TIẾP. Không fine-tune, không boost, không rescoring.

## Tại sao prune-only
Ba lần fine-tune (empty-label + 2 bản retain-aware) đều collapse: retention 1.0 → 0.02–0.20, dets ~0.07/ảnh. Bằng chứng gốc (cell 16 NB2): pseudo-GT retain set gần rỗng — 0.25 box/ảnh @conf0.2, 0.00 @conf0.5. Train FPN+head trên nhãn thưa = hiệu ứng y như empty-label = phá model. Fine-tune chính là thủ phạm.

**Giả thuyết:** pruning có thể gỡ backdoor mà GIỮ khả năng detect streak (ANP NeurIPS'21, Fine-Pruning RAID'18), vì không đụng đường dẫn detect nếu chọn đúng neuron. NB này test dứt khoát: prune-only, sweep prune_frac, đo firing/retention/val_macadd, KHÔNG train.

**Tiêu chí thắng:** tồn tại prune_frac sao cho firing↓ nhiều MÀ retention cao (≥0.6) VÀ val_macadd < baseline.

## 1. Setup

In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q ensemble_boxes

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 11.2 MB/s eta 0:00:00


In [2]:
import copy, json, itertools, pickle, random, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
from ensemble_boxes import weighted_boxes_fusion
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.data import (DatasetCatalog, DatasetMapper, MetadataCatalog,
                             build_detection_train_loader, detection_utils as utils)
from detectron2.structures import BoxMode
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
import logging; logging.getLogger('detectron2').setLevel(logging.ERROR)

ROOT = '/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models'
POISONED_WEIGHTS = f'{ROOT}/poisoned_model/poisoned_model.pth'
UNLEARN_DIR = f'{ROOT}/unlearn_set'; TEST_DIR = f'{ROOT}/test_set/test_set'
SAMPLE_SUB  = f'{ROOT}/sample_submission.csv'
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
SUBMISSION_PATH = str(WORK / 'submission.csv')

BASE_CONFIG          = 'COCO-Detection/retinanet_R_50_FPN_3x.yaml'
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1; IMG_W = IMG_H = 1024; BATCH_SIZE = 4

A_FACTOR = 10.0
IOU_THRESHOLDS = np.round(np.arange(0.2, 0.91, 0.1), 2)
CAND_THRESH = 0.05
SUBMIT_THRESH = 0.2
CLEAN_CONF_FLOOR = 0.2
MATCH_IOU = 0.3
POISON_OVERLAP_IOU = 0.1   # detection trùng poison nếu IoU>=0.1 (loại khỏi retain)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Setup OK | device:', DEVICE)

Setup OK | device: cuda


## 2. Metric core — maCADD (unit-tested)

In [3]:
def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)), np.float32)
    ix1 = np.maximum(a[:,0:1], b[None,:,0]); iy1 = np.maximum(a[:,1:2], b[None,:,1])
    ix2 = np.minimum(a[:,2:3], b[None,:,2]); iy2 = np.minimum(a[:,3:4], b[None,:,3])
    iw = np.clip(ix2-ix1,0,None); ih = np.clip(iy2-iy1,0,None); inter = iw*ih
    aa = ((a[:,2]-a[:,0])*(a[:,3]-a[:,1]))[:,None]; ab = ((b[:,2]-b[:,0])*(b[:,3]-b[:,1]))[None,:]
    return np.where(aa+ab-inter>0, inter/(aa+ab-inter+1e-9), 0.0).astype(np.float32)

def acadd_t(cb, cs, pb, ps, t, cip):
    ious = iou_matrix(cb, pb); cost = 0.0; mc=set(); mp=set()
    if ious.size:
        ri, ci = linear_sum_assignment(-ious)
        for i, j in zip(ri, ci):
            if ious[i,j] >= t:
                mc.add(i); mp.add(j); d = float(cs[i])-float(ps[j])
                if cip[i]: cost += (d/A_FACTOR) if d>0 else -d
                else:      cost += d if d>0 else (-d/A_FACTOR)
    cost += sum(float(cs[i]) for i in range(len(cs)) if i not in mc)
    cost += sum(float(ps[j]) for j in range(len(ps)) if j not in mp)
    return cost

def macadd(clean_ref, pred, poison_flags=None):
    w = IOU_THRESHOLDS/IOU_THRESHOLDS.sum(); per=[]
    for key,(cb,cs) in clean_ref.items():
        keep = cs>CLEAN_CONF_FLOOR; cb2,cs2 = cb[keep], cs[keep]
        cip = (poison_flags.get(key, np.zeros(len(cb),bool))[keep] if poison_flags else np.zeros(len(cs2),bool))
        pb,ps = pred.get(key,(np.zeros((0,4),np.float32),np.zeros(0,np.float32)))
        per.append(sum(wi*acadd_t(cb2,cs2,pb,ps,t,cip) for wi,t in zip(w,IOU_THRESHOLDS)))
    return float(np.mean(per)) if per else 0.0

bx = lambda x,y,w,h: np.array([[x,y,x+w,y+h]],np.float32); E=(np.zeros((0,4),np.float32),np.zeros(0,np.float32))
cl = {'i':(bx(10,10,40,40), np.array([0.6],np.float32))}
assert abs(macadd(cl,{'i':E})-0.6)<1e-6                                   # miss streak = FN full
assert abs(macadd(cl,{'i':(bx(10,10,40,40),np.array([0.2]))})-0.4)<1e-6   # undershoot conf = full
assert abs(macadd({'i':E},{'i':(bx(5,5,20,20),np.array([0.5]))})-0.5)<1e-6 # fire on nothing = FP full
print('maCADD unit tests passed')

maCADD unit tests passed


## 3. Load data + poisoned model

In [4]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype==np.uint16: im = im.astype(np.float32)/65535.0
    im = np.clip(im*255,0,255).astype(np.float32)
    if im.ndim==2: im = np.repeat(im[:,:,None],3,axis=2)
    return im
def build_cfg(weights, score_thresh=CAND_THRESH, output_dir=None):
    cfg = get_cfg(); cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights; cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]; cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh; cfg.MODEL.DEVICE = DEVICE; cfg.INPUT.FORMAT = 'RGB'
    if output_dir: cfg.OUTPUT_DIR = str(output_dir); Path(output_dir).mkdir(parents=True, exist_ok=True)
    return cfg
def build_predictor(weights, score_thresh=CAND_THRESH): return DefaultPredictor(build_cfg(weights, score_thresh))
def predict_image(predictor, img):
    inst = predictor(img)['instances'].to('cpu')
    return inst.pred_boxes.tensor.numpy().astype(np.float32), inst.scores.numpy().astype(np.float32)

with open(Path(UNLEARN_DIR)/'annotations_coco.json') as f: coco = json.load(f)
stem_by_id = {im['id']: Path(im['file_name']).stem for im in coco['images']}
poison_boxes_by_stem = {}
for a in coco['annotations']:
    x,y,w,h = a['bbox']; poison_boxes_by_stem.setdefault(stem_by_id[a['image_id']], []).append([x,y,x+w,y+h])
poison_boxes_by_stem = {k: np.array(v,np.float32) for k,v in poison_boxes_by_stem.items()}
poison_xywh = {}
for a in coco['annotations']: poison_xywh.setdefault(a['image_id'], []).append(a['bbox'])
unlearn_files = sorted(Path(UNLEARN_DIR).glob('*.png'), key=lambda p:int(p.stem))
unlearn_imgs = {p.stem: load_image(p) for p in unlearn_files}
test_files = sorted(Path(TEST_DIR).glob('*.png'), key=lambda p:int(p.stem))
poisoned_predictor = build_predictor(POISONED_WEIGHTS, CAND_THRESH)
print('unlearn:', len(unlearn_files), '| test:', len(test_files))

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


unlearn: 20 | test: 2000


## 4. Validation set (poison + synth streak thật)

In [5]:
MARGIN = 6
poison_patches = []
for a in coco['annotations']:
    x,y,w,h = a['bbox']; stem = stem_by_id[a['image_id']]
    x0,y0 = int(max(0,x-MARGIN)), int(max(0,y-MARGIN)); x1,y1 = int(min(IMG_W,x+w+MARGIN)), int(min(IMG_H,y+h+MARGIN))
    poison_patches.append(unlearn_imgs[stem][y0:y1, x0:x1].copy())
stems = list(unlearn_imgs)
def rand_free_spot(stem, ph, pw, rng, tries=50):
    pb = poison_boxes_by_stem.get(stem, np.array([[0,0,1,1]],np.float32))
    for _ in range(tries):
        x0 = rng.integers(16, IMG_W-pw-16); y0 = rng.integers(16, IMG_H-ph-16)
        if iou_matrix(np.array([[x0,y0,x0+pw,y0+ph]],np.float32), pb).max()<0.01: return int(x0),int(y0)
    return None
def paste_patch(img, patch, x0, y0, feather=3):
    ph,pw = patch.shape[:2]; reg = img[y0:y0+ph, x0:x0+pw]
    if reg.shape[:2]!=(ph,pw): return
    mask = np.ones((ph,pw,1),np.float32)
    if ph>2*feather and pw>2*feather:
        rh = np.minimum(np.arange(ph)[:,None]+1, np.arange(ph)[::-1][:,None]+1)/feather
        rw = np.minimum(np.arange(pw)[None,:]+1, np.arange(pw)[::-1][None,:]+1)/feather
        mask = np.clip(np.minimum(rh,rw),0,1)[...,None]
    img[y0:y0+ph, x0:x0+pw] = reg*(1-mask) + patch*mask
def synth_streak_on(bg, rng):
    h,w = bg.shape; canvas = np.zeros_like(bg)
    ang = rng.uniform(0,np.pi); L = rng.uniform(20,110); cx,cy = rng.uniform(0.25,0.75)*w, rng.uniform(0.25,0.75)*h
    p1 = (int(cx-L/2*np.cos(ang)), int(cy-L/2*np.sin(ang))); p2 = (int(cx+L/2*np.cos(ang)), int(cy+L/2*np.sin(ang)))
    cv2.line(canvas,p1,p2,1.0,1); canvas = cv2.GaussianBlur(canvas,(0,0),rng.uniform(0.6,1.6))
    amp = rng.uniform(3,14)*(np.std(bg)+1e-3); out = bg + canvas/(canvas.max()+1e-6)*amp
    box = [min(p1[0],p2[0])-3, min(p1[1],p2[1])-3, max(p1[0],p2[0])+3, max(p1[1],p2[1])+3]
    return out, [float(np.clip(box[0],0,w)),float(np.clip(box[1],0,h)),float(np.clip(box[2],0,w)),float(np.clip(box[3],0,h))]
def build_validation_set(n=80, seed=SEED):
    rng = np.random.default_rng(seed); val=[]
    for _ in range(n):
        stem = stems[rng.integers(len(stems))]; img = unlearn_imgs[stem].copy(); streaks=[]
        for _ in range(rng.integers(1,4)):
            g2,box = synth_streak_on(img[:,:,0], rng); img = np.repeat(g2[:,:,None],3,axis=2); streaks.append(box)
        pois = [list(map(float,b)) for b in poison_boxes_by_stem.get(stem, [])]
        for _ in range(rng.integers(0,3)):
            patch = poison_patches[rng.integers(len(poison_patches))]; ph,pw = patch.shape[:2]
            spot = rand_free_spot(stem,ph,pw,rng)
            if spot: x0,y0 = spot; paste_patch(img,patch,x0,y0); pois.append([x0+MARGIN,y0+MARGIN,x0+pw-MARGIN,y0+ph-MARGIN])
        val.append({'img':img,'streak_boxes':streaks,'poison_boxes':pois})
    return val
VAL = build_validation_set(80)
print('val images:', len(VAL), '| streaks:', sum(len(v['streak_boxes']) for v in VAL), '| poison:', sum(len(v['poison_boxes']) for v in VAL))

val images: 80 | streaks: 159 | poison: 163


## 5. Clean reference = CHỈ streak thật

In [6]:
def make_clean_reference(val, predictor):
    '''clean_ref = CHỈ streak thật (khớp detection model gốc để lấy conf tham chiếu).
    Poison KHÔNG vào ref -> mọi detection trên poison sẽ là FP.'''
    ref = {}
    for i,item in enumerate(val):
        b,s = predict_image(predictor, item['img'])
        rb,rs = [],[]
        sb = np.array(item['streak_boxes'],np.float32).reshape(-1,4)
        for k in range(len(sb)):
            if len(b):
                ious = iou_matrix(sb[k:k+1], b)[0]; j = ious.argmax()
                if ious[j]>=MATCH_IOU: rb.append(b[j]); rs.append(float(s[j]))
        ref[str(i)] = (np.array(rb,np.float32).reshape(-1,4), np.array(rs,np.float32))
    return ref
CLEAN_REF = make_clean_reference(VAL, poisoned_predictor)
print('clean reference (streak-only) built for', len(CLEAN_REF), 'val images')

clean reference (streak-only) built for 80 val images


## 6. Thước đo genuine + TTA helpers

In [7]:
def eval_model_on_val(predictor, val, tta=False):
    fired=ft=0; kept=st=0; pk={}
    for i,item in enumerate(val):
        b,s = (predict_tta(predictor, item['img']) if tta else predict_image(predictor, item['img']))
        keep = s>SUBMIT_THRESH; b2,s2 = (b[keep], s[keep]) if len(b) else (b,s); pk[str(i)]=(b2,s2)
        for pbx in item['poison_boxes']:
            ft+=1
            if len(b2) and iou_matrix(np.array([pbx],np.float32),b2).max()>=MATCH_IOU: fired+=1
        for sbx in item['streak_boxes']:
            st+=1
            if len(b2) and iou_matrix(np.array([sbx],np.float32),b2).max()>=MATCH_IOU: kept+=1
    return dict(poison_firing_rate=fired/max(ft,1), streak_retention=kept/max(st,1), val_macadd=macadd(CLEAN_REF, pk))

base_stats = eval_model_on_val(poisoned_predictor, VAL)
print('POISONED (chưa xử lý):', {k: round(v,4) for k,v in base_stats.items()})
print('-> baseline: firing cao (bắn poison=FP), retention=1, val_macadd = mức của model gốc.')
print('   Mục tiêu: firing↓ (bớt FP poison) mà retention giữ -> val_macadd < baseline.')

POISONED (chưa xử lý): {'poison_firing_rate': 0.9693, 'streak_retention': 1.0, 'val_macadd': 1.4549}
-> baseline: firing cao (bắn poison=FP), retention=1, val_macadd = mức của model gốc.
   Mục tiêu: firing↓ (bớt FP poison) mà retention giữ -> val_macadd < baseline.


In [8]:
D4 = [(k,f) for f in (0,1) for k in range(4)]
def apply_d4(img,k,flip):
    x = np.rot90(img,k).copy() if k else img
    return x[:, ::-1, :].copy() if flip else x
def invert_d4(boxes,k,flip,H,W):
    S = W; b = boxes.astype(np.float32).copy()
    if flip: x1 = S-b[:,2]; x2 = S-b[:,0]; b[:,0],b[:,2] = x1,x2
    for _ in range(k%4):
        x1=b[:,0].copy(); y1=b[:,1].copy(); x2=b[:,2].copy(); y2=b[:,3].copy()
        b[:,0]=S-y2; b[:,1]=x1; b[:,2]=S-y1; b[:,3]=x2
    return b
def predict_tta(predictor, img):
    H,W = img.shape[:2]; bl,sl,ll = [],[],[]
    for k,flip in D4:
        b,s = predict_image(predictor, apply_d4(img,k,flip))
        if len(b)==0: continue
        b = invert_d4(b,k,flip,H,W); bn = b.copy()
        bn[:,[0,2]] = np.clip(bn[:,[0,2]],0,W)/W; bn[:,[1,3]] = np.clip(bn[:,[1,3]],0,H)/H
        bl.append(bn.tolist()); sl.append(s.tolist()); ll.append([0]*len(s))
    if not bl: return np.zeros((0,4),np.float32), np.zeros(0,np.float32)
    fb,fs,_ = weighted_boxes_fusion(bl,sl,ll,iou_thr=0.55,skip_box_thr=CAND_THRESH)
    fb = fb.copy(); fb[:,[0,2]]*=W; fb[:,[1,3]]*=H
    return fb.astype(np.float32), fs.astype(np.float32)


## 7. Activation-guided channel pruning (cls_subnet)

In [9]:
def collect_and_prune(model, prune_frac):
    model.eval()
    layers = [m for m in model.head.cls_subnet if isinstance(m, nn.Conv2d)]
    stored = {i: [] for i in range(len(layers))}; hooks=[]
    for i,ly in enumerate(layers):
        hooks.append(ly.register_forward_hook(lambda m,inp,out,idx=i: stored[idx].append(out.detach().cpu())))
    dicts = [{'file_name': str(Path(UNLEARN_DIR)/f'{s}.png'), 'image_id': int(s), 'height': IMG_H, 'width': IMG_W} for s in unlearn_imgs]
    with torch.no_grad():
        for d in dicts:
            im = load_image(d['file_name']); model([{'image': torch.as_tensor(im.transpose(2,0,1)).to(DEVICE)}])
    for h in hooks: h.remove()
    conv_layers = [(i,m) for i,m in enumerate(model.head.cls_subnet) if isinstance(m, nn.Conv2d)]
    total = 0
    for li,(mi,layer) in enumerate(conv_layers):
        acts = stored[li]; C = acts[0].shape[1]; fg = torch.zeros(C); bg = torch.zeros(C); nf=nb_=0
        rng = np.random.default_rng(SEED)
        for act,d in zip(acts, dicts):
            a = act[0]; _,aH,aW = a.shape; sy,sx = aH/d['height'], aW/d['width']
            for (x,y,w,h) in poison_xywh.get(d['image_id'], []):
                x1,y1 = max(0,int(x*sx)),max(0,int(y*sy)); x2,y2 = min(aW,int((x+w)*sx)+1),min(aH,int((y+h)*sy)+1)
                if x2>x1 and y2>y1: fg += a[:,y1:y2,x1:x2].relu().mean([1,2]); nf+=1
            for _ in range(max(1,len(poison_xywh.get(d['image_id'],[])))):
                ph,pw = max(1,int(16*sy)),max(1,int(16*sx)); ry,rx = rng.integers(0,max(1,aH-ph)),rng.integers(0,max(1,aW-pw))
                bg += a[:,ry:ry+ph,rx:rx+pw].relu().mean([1,2]); nb_+=1
        if nf: fg/=nf
        if nb_: bg/=nb_
        score = (fg-bg).numpy(); n_prune = max(1,int(len(score)*prune_frac)); bad = np.argsort(score)[-n_prune:]
        with torch.no_grad():
            layer.weight.data[bad] = 0.0
            if layer.bias is not None: layer.bias.data[bad] = 0.0
        total += n_prune
    return model, total

## 8. Baseline: model poisoned (chưa xử lý)

In [10]:
base_stats = eval_model_on_val(poisoned_predictor, VAL)
print('POISONED (baseline):', {k: round(v,4) for k,v in base_stats.items()})
print('-> firing cao (poison=FP), retention=1.0, val_macadd = mức model gốc.')
print('   Mục tiêu prune-only: firing↓ mà retention giữ ≥0.6 -> val_macadd < baseline.')

POISONED (baseline): {'poison_firing_rate': 0.9693, 'streak_retention': 1.0, 'val_macadd': 1.4549}
-> firing cao (poison=FP), retention=1.0, val_macadd = mức model gốc.
   Mục tiêu prune-only: firing↓ mà retention giữ ≥0.6 -> val_macadd < baseline.


## 9. Prune-only: prune neuron nhạy poison rồi LƯU weight (KHÔNG fine-tune)

In [11]:
def make_pruned_model(prune_frac, tag):
    out = WORK/tag; out.mkdir(parents=True, exist_ok=True)
    cfg0 = build_cfg(POISONED_WEIGHTS, output_dir=out)
    model = build_model(cfg0); DetectionCheckpointer(model).load(POISONED_WEIGHTS); model.to(DEVICE)
    n_pruned = 0
    if prune_frac > 0:
        model, n_pruned = collect_and_prune(model, prune_frac)
    w = str(out/'pruned.pth'); torch.save({'model': model.state_dict()}, w)
    del model; torch.cuda.empty_cache()
    return w, n_pruned

## 10. Sweep prune_frac (chỉ prune, đo trên validation)

In [12]:
PRUNE_SWEEP = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
prune_results = []
for pf in PRUNE_SWEEP:
    tag = f'p{int(pf*100):02d}'
    w, n = make_pruned_model(pf, tag)
    pr = build_predictor(w, CAND_THRESH); stats = eval_model_on_val(pr, VAL); del pr; torch.cuda.empty_cache()
    prune_results.append({'tag': tag, 'prune_frac': pf, 'n_pruned': n, 'weights': w, **stats})
    print(f"{tag}: prune={pf:.2f} nchan={n} firing={stats['poison_firing_rate']:.3f} "
          f"retention={stats['streak_retention']:.3f} val_macadd={stats['val_macadd']:.4f}")
prune_df = pd.DataFrame(prune_results)
print('\n', prune_df[['tag','prune_frac','n_pruned','poison_firing_rate','streak_retention','val_macadd']].to_string(index=False))
print(f"\nPOISONED baseline: firing={base_stats['poison_firing_rate']:.3f} "
      f"retention={base_stats['streak_retention']:.3f} val_macadd={base_stats['val_macadd']:.4f}")

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


p00: prune=0.00 nchan=0 firing=0.969 retention=1.000 val_macadd=1.4549


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


p05: prune=0.05 nchan=48 firing=0.282 retention=0.553 val_macadd=1.0930


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


...

p30: prune=0.30 nchan=304 firing=0.000 retention=0.000 val_macadd=1.3069


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


p40: prune=0.40 nchan=408 firing=0.000 retention=0.000 val_macadd=1.3069


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


p50: prune=0.50 nchan=512 firing=0.000 retention=0.000 val_macadd=1.3069

 tag  prune_frac  n_pruned  poison_firing_rate  streak_retention  val_macadd
p00        0.00         0            0.969325          1.000000    1.454948
p05        0.05        48            0.282209          0.553459    1.092997
p10        0.10       100            0.018405          0.201258    1.205367
p15        0.15       152            0.000000          0.037736    1.287461
p20        0.20       204            0.000000          0.012579    1.300227
p25        0.25       256            0.000000          0.000000    1.306936
p30        0.30       304            0.000000          0.000000    1.306936
p40        0.40       408            0.000000          0.000000    1.306936
p50        0.50       512            0.000000          0.000000    1.306936

POISONED baseline: firing=0.969 retention=1.000 val_macadd=1.4549


## 11. Chọn cấu hình thắng (retention cao + val_macadd thấp nhất)

In [13]:
RETENTION_FLOOR = 0.6
valid = [r for r in prune_results if r['streak_retention'] >= RETENTION_FLOOR
         and r['val_macadd'] < base_stats['val_macadd'] and r['prune_frac'] > 0]
if valid:
    best = min(valid, key=lambda r: r['val_macadd'])
    PRUNE_ONLY_WINS = True
    print('PRUNE-ONLY THẮNG:', {k: (round(v,4) if isinstance(v,float) else v)
          for k,v in best.items() if k!='weights'})
else:
    PRUNE_ONLY_WINS = False
    best = min([r for r in prune_results if r['prune_frac']>0], key=lambda r: r['val_macadd'])
    print('KHÔNG cấu hình prune-only nào đạt (retention>=0.6 & val_macadd<baseline).')
    print('Ứng viên tốt nhất (tham khảo):', {k: (round(v,4) if isinstance(v,float) else v)
          for k,v in best.items() if k!='weights'})
    print('-> Nếu retention rớt cùng lúc với firing => backdoor bám chung neuron detect;')
    print('   chuyển sang Bước 2 (fine-tune nhẹ với retain set DÀY từ ảnh test) hoặc Neural Cleanse.')
FINAL_WEIGHTS = best['weights']; FINAL_STATS = {k:best[k] for k in ('poison_firing_rate','streak_retention','val_macadd')}

KHÔNG cấu hình prune-only nào đạt (retention>=0.6 & val_macadd<baseline).
Ứng viên tốt nhất (tham khảo): {'tag': 'p05', 'prune_frac': 0.05, 'n_pruned': 48, 'poison_firing_rate': 0.2822, 'streak_retention': 0.5535, 'val_macadd': 1.093}
-> Nếu retention rớt cùng lúc với firing => backdoor bám chung neuron detect;
   chuyển sang Bước 2 (fine-tune nhẹ với retain set DÀY từ ảnh test) hoặc Neural Cleanse.


## 12. TTA D4+WBF (tùy chọn — chỉ nhận nếu cải thiện validation)

In [14]:
final_pr = build_predictor(FINAL_WEIGHTS, CAND_THRESH)
tta_stats = eval_model_on_val(final_pr, VAL, tta=True)
print('no-TTA :', {k: round(float(v),4) for k,v in FINAL_STATS.items()})
print('TTA    :', {k: round(v,4) for k,v in tta_stats.items()})
USE_TTA = tta_stats['val_macadd'] <= FINAL_STATS['val_macadd']
print('-> dùng TTA:', USE_TTA)

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


no-TTA : {'poison_firing_rate': 0.2822, 'streak_retention': 0.5535, 'val_macadd': 1.093}
TTA    : {'poison_firing_rate': 0.2945, 'streak_retention': 0.5723, 'val_macadd': 1.1365}
-> dùng TTA: False


## 13. Submission = output trực tiếp model đã prune

In [15]:
sample = pd.read_csv(SAMPLE_SUB, dtype={'image_id': str})
def fmt(b, s):
    parts = []
    for (x1,y1,x2,y2), sc in zip(b, s):
        x1,y1 = float(np.clip(x1,0,IMG_W)), float(np.clip(y1,0,IMG_H))
        x2,y2 = float(np.clip(x2,0,IMG_W)), float(np.clip(y2,0,IMG_H))
        w,h = x2-x1, y2-y1
        if w>0 and h>0 and sc>0: parts += [f'{float(sc):.6f}', f'{x1:.2f}', f'{y1:.2f}', f'{w:.2f}', f'{h:.2f}']
    return ' '.join(parts) if parts else ' '
pred = {}
for p in tqdm(test_files, desc='infer test'):
    img = load_image(p)
    b,s = predict_tta(final_pr, img) if USE_TTA else predict_image(final_pr, img)
    keep = s > SUBMIT_THRESH; pred[p.stem] = (b[keep], s[keep])
sub = sample.copy()
sub['prediction_string'] = sub['image_id'].map(lambda i: fmt(*pred.get(str(i), E)))
sub['prediction_string'] = sub['prediction_string'].fillna(' ')
sub.loc[sub['prediction_string'].str.strip()=='', 'prediction_string'] = ' '
sub.to_csv(SUBMISSION_PATH, index=False)
total = sum(len(s) for _,s in pred.values()); npred = int((sub['prediction_string'].str.strip()!='').sum())
print(f'saved {SUBMISSION_PATH} | rows={len(sub)} | dets={total} ({total/len(test_files):.3f}/img) | imgs w/ preds: {npred}')
print('so sánh: poisoned ~4.1/img | clean mục tiêu ~1.5/img | empty=0. Bản collapse cũ = 0.068/img.')

infer test: 100%|██████████| 2000/2000 [04:32<00:00,  7.34it/s]

saved /kaggle/working/submission.csv | rows=2000 | dets=743 (0.371/img) | imgs w/ preds: 618
so sánh: poisoned ~4.1/img | clean mục tiêu ~1.5/img | empty=0. Bản collapse cũ = 0.068/img.


## 14. Tổng kết + quy tắc nộp

In [16]:
print('='*60)
print('POISONED   :', {k: round(v,4) for k,v in base_stats.items()})
print('DE-POISONED:', {k: round(float(v),4) for k,v in FINAL_STATS.items()})
dets_per_img = total/len(test_files)
ok = PRUNE_ONLY_WINS and (0.5 <= dets_per_img <= 3.5)
print('\nĐẠT TIÊU CHÍ NỘP:', ok)
if ok:
    print(' -> Nên nộp: prune-only giữ retention & giảm firing, dets/img hợp lý (gần clean ~1.5).')
else:
    print(' -> CHƯA nộp. Prune-only chưa tách được poison khỏi streak, hoặc dets/img bất thường.')
    print('    Bước 2: fine-tune nhẹ với retain set DÀY (pseudo-GT từ 2000 ảnh test, không chỉ 20 ảnh poison).')
    print('    Bước 3: Neural Cleanse (reverse-engineer trigger) — baseline chính thức BTC, chất paper.')

POISONED   : {'poison_firing_rate': 0.9693, 'streak_retention': 1.0, 'val_macadd': 1.4549}
DE-POISONED: {'poison_firing_rate': 0.2822, 'streak_retention': 0.5535, 'val_macadd': 1.093}

ĐẠT TIÊU CHÍ NỘP: False
 -> CHƯA nộp. Prune-only chưa tách được poison khỏi streak, hoặc dets/img bất thường.
    Bước 2: fine-tune nhẹ với retain set DÀY (pseudo-GT từ 2000 ảnh test, không chỉ 20 ảnh poison).
    Bước 3: Neural Cleanse (reverse-engineer trigger) — baseline chính thức BTC, chất paper.
